# 📘 NotebookUtils (formerly MSSparkUtils) — Complete Training Guide
### Microsoft Fabric | Data Engineering

---

**NotebookUtils** is a built-in utility package available in every Microsoft Fabric Notebook. It provides a rich set of helpers for:

| Module | Purpose |
|---|---|
| `notebookutils.fs` | File system operations (ADLS Gen2, OneLake, local) |
| `notebookutils.notebook` | Notebook orchestration and chaining |
| `notebookutils.credentials` | Token retrieval and Azure Key Vault secrets |
| `notebookutils.variableLibrary` | Access centrally managed variable libraries |
| `notebookutils.lakehouse` | Lakehouse artifact management |
| `notebookutils.runtime` | Session context and metadata |
| `notebookutils.session` | Session lifecycle control |

> ⚠️ **Rename Notice:** `mssparkutils` has been officially renamed to `notebookutils`. The old namespace remains backward-compatible but will be retired in a future release. Always use `notebookutils` in new code.

> ✅ **Runtime Requirement:** NotebookUtils requires **Spark 3.4 (Runtime v1.2) and above**.

## 📋 Table of Contents

1. [Getting Help](#1-getting-help)
2. [File System Utilities — `notebookutils.fs`](#2-file-system-utilities)
   - 2.1 List Files
   - 2.2 View File Properties
   - 2.3 Create Directory
   - 2.4 Copy File
   - 2.5 Fast Copy File
   - 2.6 Preview File Content
   - 2.7 Move File
   - 2.8 Write File
   - 2.9 Append to File
   - 2.10 Delete File or Directory
   - 2.11 Check File Existence
   - 2.12 Mount / Unmount Storage
3. [Notebook Utilities — `notebookutils.notebook`](#3-notebook-utilities)
   - 3.1 Run a Notebook
   - 3.2 Run Multiple Notebooks in Parallel (DAG)
   - 3.3 Exit a Notebook with a Value
   - 3.4 Manage Notebook Artifacts
4. [Credentials Utilities — `notebookutils.credentials`](#4-credentials-utilities)
   - 4.1 Get Token
   - 4.2 Get Secret from Key Vault
5. [Variable Library Utilities — `notebookutils.variableLibrary`](#5-variable-library-utilities)
6. [Lakehouse Utilities — `notebookutils.lakehouse`](#6-lakehouse-utilities)
7. [Runtime Utilities — `notebookutils.runtime`](#7-runtime-utilities)
8. [Session Utilities — `notebookutils.session`](#8-session-utilities)
9. [Quick Reference Cheat Sheet](#9-quick-reference-cheat-sheet)

---
## 1. Getting Help

Every module in NotebookUtils has a built-in `help()` method. This is the first thing to run when you want to explore available methods.

In [ ]:
# Top-level help — shows all available modules
notebookutils.help()

In [ ]:
# Help for a specific module
notebookutils.fs.help()

In [ ]:
# Help for a specific method within a module
notebookutils.fs.help("cp")

In [ ]:
# Help for the notebook module
notebookutils.notebook.help()

---
## 2. File System Utilities — `notebookutils.fs`

`notebookutils.fs` provides utilities to work with file systems including:
- **OneLake / ADLS Gen2** (`abfss://` paths)
- **Azure Blob Storage**
- **Local driver node file system** (`file:/` paths)
- **Default Lakehouse** (relative paths like `Files/...`)

### 📌 Path Reference

| Scenario | Example Path |
|---|---|
| Default Lakehouse (Spark notebook) | `Files/myfolder/myfile.csv` |
| Default Lakehouse (Python notebook) | `/lakehouse/default/Files/myfolder/myfile.csv` |
| ADLS Gen2 / OneLake absolute | `abfss://<container>@<account>.dfs.core.windows.net/<path>` |
| Local driver file system | `file:/tmp/myfile.txt` |

> ⚠️ In **Python notebooks**, relative paths resolve to the local working directory — always use full `abfss://` or `/lakehouse/default/...` paths.

### 2.1 List Files — `notebookutils.fs.ls()`

Lists the contents of a directory. Returns an array of file info objects.

In [ ]:
# List files in the default Lakehouse 'Files' folder (Spark notebook)
notebookutils.fs.ls("Files/")

In [ ]:
# List files using absolute ABFSS path
notebookutils.fs.ls("abfss://<container_name>@<storage_account_name>.dfs.core.windows.net/<path>")

In [ ]:
# List files in the local driver file system
notebookutils.fs.ls("file:/tmp")

### 2.2 View File Properties

Each item returned by `ls()` has these properties:
- `name` — file or directory name
- `path` — full path
- `size` — size in bytes
- `isDir` — True if it's a directory
- `isFile` — True if it's a file

In [ ]:
# Iterate and print file properties
files = notebookutils.fs.ls("Files/")

for file in files:
    print(f"Name: {file.name}")
    print(f"  Path:   {file.path}")
    print(f"  Size:   {file.size} bytes")
    print(f"  isDir:  {file.isDir}")
    print(f"  isFile: {file.isFile}")
    print()

### 2.3 Create Directory — `notebookutils.fs.mkdirs()`

Creates the given directory if it does not exist. Also creates any necessary parent directories (like `mkdir -p` in Linux).

In [ ]:
# Create a folder in the default Lakehouse (Spark notebook — relative path)
notebookutils.fs.mkdirs("Files/training/output")

In [ ]:
# Create a folder using an absolute ABFSS path
notebookutils.fs.mkdirs("abfss://<container_name>@<storage_account_name>.dfs.core.windows.net/new_folder")

In [ ]:
# Create a folder in the local driver file system
notebookutils.fs.mkdirs("file:/tmp/my_local_dir")

### 2.4 Copy File — `notebookutils.fs.cp()`

Copies a file or directory, and supports copying **across file systems** (e.g., from ADLS to OneLake). Use `recurse=True` to copy entire directories.

In [ ]:
# Copy a single file within the default Lakehouse
notebookutils.fs.cp(
    "Files/raw/myfile.csv",        # source
    "Files/archive/myfile.csv"     # destination
)

In [ ]:
# Copy an entire directory recursively
notebookutils.fs.cp(
    "Files/raw/",
    "Files/backup/raw/",
    recurse=True    # copies all files and subdirectories
)

In [ ]:
# Copy across file systems (ADLS Gen2 to OneLake)
notebookutils.fs.cp(
    "abfss://source-container@source-account.dfs.core.windows.net/myfile.csv",
    "abfss://destination-container@onelake.dfs.fabric.microsoft.com/myfile.csv",
    recurse=False
)

### 2.5 Fast Copy File — `notebookutils.fs.fastcp()`

A higher-performance copy method that uses **AzCopy** under the hood. Recommended over `cp()` when dealing with **large data volumes**.

> ⚠️ `fastcp()` does **not** support copying files across OneLake regions. Use `cp()` for cross-region scenarios.

In [ ]:
# Fast copy — preferred for large datasets
notebookutils.fs.fastcp(
    "Files/large_dataset/",
    "Files/backup/large_dataset/",
    recurse=True
)

### 2.6 Preview File Content — `notebookutils.fs.head()`

Returns the first `maxBytes` bytes of a file as a UTF-8 string. Useful for quickly inspecting file content without fully reading it.

In [ ]:
# Preview the first 1024 bytes (default) of a file
content_preview = notebookutils.fs.head("Files/raw/myfile.csv")
print(content_preview)

In [ ]:
# Preview more bytes — e.g., first 5000 bytes
content_preview = notebookutils.fs.head("Files/raw/myfile.csv", 5000)
print(content_preview)

### 2.7 Move File — `notebookutils.fs.mv()`

Moves a file or directory. Supports moves across file systems. Parameters:
- **3rd param** (`createPath`): If `True`, creates parent directories if they don't exist
- **4th param** (`overwrite`): If `True`, overwrites the destination if it already exists

In [ ]:
# Move a file — creates parent directory if needed
notebookutils.fs.mv(
    "Files/raw/myfile.csv",         # source
    "Files/processed/myfile.csv",   # destination
    True                            # createPath = True
)

In [ ]:
# Move and overwrite at destination
notebookutils.fs.mv(
    "Files/raw/myfile.csv",
    "Files/processed/myfile.csv",
    True,   # createPath
    True    # overwrite
)

### 2.8 Write File — `notebookutils.fs.put()`

Writes a string to a file, encoded in UTF-8. If the file exists, set `overwrite=True` to replace it.

> ⚠️ Does not support concurrent writes to the same file.

In [ ]:
# Write a new file
notebookutils.fs.put(
    "Files/output/hello.txt",   # file path
    "Hello from Fabric!",       # content
    True                        # overwrite = True
)

In [ ]:
# Practical example: write a JSON config file
import json

config = {
    "pipeline": "pl_load_sftp",
    "client": "TUFTS",
    "version": "1.0"
}

notebookutils.fs.put(
    "Files/config/pipeline_config.json",
    json.dumps(config, indent=2),
    True
)

### 2.9 Append to File — `notebookutils.fs.append()`

Appends a string to a file. If the file does not exist, set `createFileIfNotExists=True` to create it.

> ⚠️ When calling `append()` in a loop on the same file, add a **0.5–1 second sleep** between writes. The internal `flush` is asynchronous, and rapid writes can cause data integrity issues.

In [ ]:
# Append a line to an existing log file
notebookutils.fs.append(
    "Files/logs/pipeline.log",          # file path
    "2024-01-15 10:30:00 | INFO | Pipeline started\n",  # content
    True                                 # createFileIfNotExists
)

In [ ]:
# Appending in a loop — always add a sleep to ensure data integrity
import time

log_entries = [
    "Step 1: Ingestion complete\n",
    "Step 2: Transformation complete\n",
    "Step 3: Load complete\n"
]

for entry in log_entries:
    notebookutils.fs.append("Files/logs/pipeline.log", entry, True)
    time.sleep(0.5)   # ← Required when appending in a loop

### 2.10 Delete File or Directory — `notebookutils.fs.rm()`

Removes a file or directory. Use `recurse=True` to delete a directory and all of its contents.

In [ ]:
# Delete a single file
notebookutils.fs.rm("Files/temp/myfile.csv")

In [ ]:
# Delete an entire directory and all its contents
notebookutils.fs.rm(
    "Files/temp/",
    recurse=True    # required to delete non-empty directories
)

### 2.11 Check File Existence — `notebookutils.fs.exists()`

Returns `True` if the file or directory exists, `False` otherwise. Very useful for conditional logic in pipelines.

In [ ]:
# Check if a file exists
file_path = "Files/raw/myfile.csv"

if notebookutils.fs.exists(file_path):
    print(f"✅ File found: {file_path}")
else:
    print(f"❌ File not found: {file_path}")

In [ ]:
# Practical pattern: create output directory only if it doesn't exist
output_dir = "Files/processed/2024/01/"

if not notebookutils.fs.exists(output_dir):
    notebookutils.fs.mkdirs(output_dir)
    print(f"📁 Created directory: {output_dir}")
else:
    print(f"📁 Directory already exists: {output_dir}")

### 2.12 Mount / Unmount Storage — `notebookutils.fs.mount()`

Mounts a remote storage directory (ADLS Gen2, Blob Storage, or a Lakehouse) to a local mount point. After mounting, you can access files using standard Python file I/O or local paths.

**Authentication options:** `accountKey` or `sasToken` (store in Azure Key Vault — never hardcode!)

**Key parameters:**
- `fileCacheTimeout` (default: 120s) — how long files are cached locally before re-fetching from the server
- `timeout` (default: 120s) — mount operation timeout

> ⚠️ Mount points are **job-level** (not persisted between sessions). Always call `unmount()` explicitly at the end of your notebook to release resources.

In [ ]:
# Mount using Account Key (retrieved from Key Vault — never hardcode the key!)
accountKey = notebookutils.credentials.getSecret(
    "https://<your-keyvault>.vault.azure.net/",
    "<secret-name>"
)

notebookutils.fs.mount(
    "abfss://mycontainer@<accountname>.dfs.core.windows.net",  # remote storage
    "/mymount",                                                 # local mount point
    {"accountKey": accountKey}
)

In [ ]:
# Mount using SAS Token
sasToken = notebookutils.credentials.getSecret(
    "https://<your-keyvault>.vault.azure.net/",
    "<sas-secret-name>"
)

notebookutils.fs.mount(
    "abfss://mycontainer@<accountname>.dfs.core.windows.net",
    "/mymount",
    {"sasToken": sasToken}
)

In [ ]:
# Mount a Fabric Lakehouse directly
notebookutils.fs.mount(
    "abfss://<workspace_name>@onelake.dfs.fabric.microsoft.com/<lakehouse_name>.Lakehouse",
    "/mylakehouse"
)

In [ ]:
# Mount with custom cache timeout (set to 0 to always fetch latest file)
notebookutils.fs.mount(
    "abfss://mycontainer@<accountname>.dfs.core.windows.net",
    "/freshdata",
    {
        "accountKey": accountKey,
        "fileCacheTimeout": 0,   # always get the latest version
        "timeout": 200           # increase if mount times out on large clusters
    }
)

In [ ]:
# Access files under the mount point
# Use getMountPath() to get the exact local path
mount_path = notebookutils.fs.getMountPath("/mymount")
print(f"Local mount path: {mount_path}")

# List files via notebookutils.fs
notebookutils.fs.ls(f"file://{mount_path}")

# Or read directly with Python standard file I/O
with open(f"{mount_path}/myfile.txt", "r") as f:
    print(f.read())

In [ ]:
# List all current mount points
notebookutils.fs.mounts()

In [ ]:
# Unmount when done — always do this explicitly at the end of your notebook
notebookutils.fs.unmount("/mymount")
print("✅ Unmounted /mymount")

---
## 3. Notebook Utilities — `notebookutils.notebook`

These utilities let you **chain notebooks together**, pass parameters between them, run them in parallel, and manage notebook artifacts programmatically.

### 3.1 Run a Notebook — `notebookutils.notebook.run()`

Runs a child notebook and returns its exit value. The child notebook runs on the same Spark pool as the parent.

**Signature:** `run(path, timeoutSeconds, arguments, workspaceId)`

> ℹ️ You can view a **snapshot link** in cell output to inspect the child notebook's run results.

In [ ]:
# Run a notebook with default parameters
exit_value = notebookutils.notebook.run(
    "nb_transform_data",   # notebook name (same workspace)
    90                     # timeout in seconds
)
print(f"Child notebook returned: {exit_value}")

In [ ]:
# Run a notebook and pass parameters
exit_value = notebookutils.notebook.run(
    "nb_process_file",
    120,                                    # timeout
    {"file_date": "2024-01-15",
     "client_code": "TUFTS",
     "max_retries": 3}
)
print(f"Result: {exit_value}")

In [ ]:
# Run a notebook from a DIFFERENT workspace (requires Runtime v1.2+)
exit_value = notebookutils.notebook.run(
    "nb_shared_utility",
    90,
    {"input": "value"},
    "fe0a6e2a-a909-4aa3-a698-0a651de790aa"   # workspace ID
)
print(f"Result: {exit_value}")

### 3.2 Run Multiple Notebooks in Parallel — `notebookutils.notebook.runMultiple()`

Runs multiple notebooks simultaneously, or in a predefined dependency order (DAG — Directed Acyclic Graph). All notebooks share compute resources within the same Spark session.

**DAG configuration options:**
- `name` — unique activity name
- `path` — notebook name
- `args` — parameters to pass
- `dependencies` — list of activity names that must complete first
- `retry` — number of retries on failure
- `retryIntervalInSeconds` — wait between retries
- `timeoutPerCellInSeconds` — per-cell timeout (default: 90s)
- `timeoutInSeconds` — overall DAG timeout (default: 12 hours)
- `concurrency` — max notebooks running at once (default: 50 for Spark, 25 for Python)

In [ ]:
# Simple parallel run — all notebooks start at the same time
notebookutils.notebook.runMultiple([
    "nb_process_client_A",
    "nb_process_client_B",
    "nb_process_client_C"
])

In [ ]:
# Advanced DAG orchestration with dependencies
# In this example:
#   - Ingest_A and Ingest_B run in parallel
#   - Transform runs only after BOTH ingestion steps complete

DAG = {
    "activities": [
        {
            "name": "Ingest_A",
            "path": "nb_ingest_source_a",
            "timeoutPerCellInSeconds": 120,
            "args": {"source": "ADLS", "date": "2024-01-15"}
        },
        {
            "name": "Ingest_B",
            "path": "nb_ingest_source_b",
            "timeoutPerCellInSeconds": 120,
            "args": {"source": "SFTP", "date": "2024-01-15"}
        },
        {
            "name": "Transform",
            "path": "nb_transform_and_merge",
            "timeoutPerCellInSeconds": 300,
            "args": {"date": "2024-01-15"},
            "retry": 2,
            "retryIntervalInSeconds": 30,
            "dependencies": ["Ingest_A", "Ingest_B"]  # waits for both to finish
        }
    ],
    "timeoutInSeconds": 3600,   # 1 hour total timeout
    "concurrency": 10
}

results = notebookutils.notebook.runMultiple(DAG)

# Print exit values from each activity
for name, result in results.items():
    print(f"{name}: {result.exitValue}")

In [ ]:
# Validate your DAG definition before running
is_valid = notebookutils.notebook.validateDAG(DAG)
print(f"DAG is valid: {is_valid}")

### 3.3 Exit a Notebook — `notebookutils.notebook.exit()`

Exits the current notebook and passes a return value to the caller (parent notebook or pipeline).

**Behavior varies by context:**
- **Interactive run:** Throws an exception and stops subsequent cells (Spark session stays alive)
- **Pipeline run:** Notebook activity completes with the exit value; Spark session stops
- **Reference run (called via `run()`):** Stops the child notebook; parent continues from the next cell

> ⚠️ Always call `exit()` in a **dedicated cell** — it overwrites the current cell's output.

In [ ]:
# Exit with a simple string value
notebookutils.notebook.exit("SUCCESS")

In [ ]:
# Exit with a JSON payload — useful for passing structured results to a pipeline
import json

result = {
    "status": "SUCCESS",
    "rows_processed": 1500,
    "files_ingested": 5,
    "timestamp": "2024-01-15T10:30:00Z"
}

notebookutils.notebook.exit(json.dumps(result))

In [ ]:
# Typical pattern: parent notebook reads the exit value from child
exit_raw = notebookutils.notebook.run("nb_child_notebook", 120, {"date": "2024-01-15"})
exit_data = json.loads(exit_raw)

print(f"Status:          {exit_data['status']}")
print(f"Rows processed:  {exit_data['rows_processed']}")
print(f"Files ingested:  {exit_data['files_ingested']}")

### 3.4 Manage Notebook Artifacts

You can programmatically **create, read, update, delete, and list** notebook items in a workspace.

In [ ]:
# Create a new notebook artifact from file content
with open("/path/to/template_notebook.ipynb", "r") as f:
    content = f.read()

artifact = notebookutils.notebook.create(
    "nb_new_notebook",      # name
    "Auto-generated notebook",  # description
    content,                # .ipynb content as string
    "lh_visiun_rl",         # default lakehouse name
    "<lakehouse_workspace_id>"
)
print(artifact)

In [ ]:
# Get a notebook artifact by name
artifact = notebookutils.notebook.get("nb_transform_data")
print(artifact)

In [ ]:
# Get the full .ipynb definition of a notebook
definition = notebookutils.notebook.getDefinition("nb_transform_data", format="ipynb")
print(definition[:500])  # preview first 500 chars

In [ ]:
# Rename a notebook
updated = notebookutils.notebook.update(
    "nb_old_name",    # current name
    "nb_new_name",    # new name
    "Updated description"
)
print(updated)

In [ ]:
# List all notebooks in the current workspace
notebooks = notebookutils.notebook.list()
for nb in notebooks:
    print(nb.displayName)

In [ ]:
# Delete a notebook by name
is_deleted = notebookutils.notebook.delete("nb_temp_notebook")
print(f"Deleted: {is_deleted}")

---
## 4. Credentials Utilities — `notebookutils.credentials`

These utilities provide secure access to **authentication tokens** and **Azure Key Vault secrets** — essential for connecting to external services without hardcoding credentials.

### 4.1 Get Token — `notebookutils.credentials.getToken()`

Returns a Microsoft Entra (Azure AD) bearer token for a given audience. Use this for authenticated API calls.

| Audience Key | Resource |
|---|---|
| `"storage"` | Azure Data Lake Storage / Blob Storage |
| `"pbi"` | Power BI / Fabric REST API |
| `"keyvault"` | Azure Key Vault |
| `"kusto"` | Synapse RTA / KQL DB |

> ⚠️ Under a **service principal**, `getToken('pbi')` has limited scope. For full Fabric service scope, use MSAL authentication.

In [ ]:
# Get a token for Azure Storage
storage_token = notebookutils.credentials.getToken("storage")
print(f"Storage token (first 50 chars): {storage_token[:50]}...")

In [ ]:
# Get a token for Fabric / Power BI REST API calls
pbi_token = notebookutils.credentials.getToken("pbi")

# Use it in a REST API call
import requests

workspace_id = "<your-workspace-id>"
headers = {"Authorization": f"Bearer {pbi_token}"}

response = requests.get(
    f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items",
    headers=headers
)
print(response.status_code)

In [ ]:
# Use storage token for JDBC connection to Fabric Warehouse (pyodbc pattern)
token = notebookutils.credentials.getToken("storage")

jdbc_url = (
    "jdbc:sqlserver://<workspace>.datawarehouse.fabric.microsoft.com:1433;"
    "database=<warehouse_name>;"
    "encrypt=true;trustServerCertificate=false;"
    "hostNameInCertificate=*.datawarehouse.fabric.microsoft.com;"
    "loginTimeout=30;"
)

# Token-based authentication (no username/password needed)
print("JDBC URL configured with token auth ✅")

### 4.2 Get Secret — `notebookutils.credentials.getSecret()`

Fetches a secret from **Azure Key Vault** using the current user's credentials. This is the recommended way to handle passwords, connection strings, API keys, and SAS tokens in notebooks.

> ✅ **Best practice:** Never hardcode secrets in notebook code. Always retrieve them from Key Vault.

In [ ]:
# Retrieve a secret from Azure Key Vault
sftp_password = notebookutils.credentials.getSecret(
    "https://my-keyvault.vault.azure.net/",  # Key Vault URI
    "sftp-client-password"                    # Secret name
)

# Use the secret (it will be redacted in notebook output automatically)
print("SFTP password retrieved ✅")

In [ ]:
# Practical pattern: retrieve multiple secrets for a pipeline
vault_uri = "https://kv-visiun-dev.vault.azure.net/"

config = {
    "sftp_host":     notebookutils.credentials.getSecret(vault_uri, "sftp-host"),
    "sftp_user":     notebookutils.credentials.getSecret(vault_uri, "sftp-username"),
    "sftp_password": notebookutils.credentials.getSecret(vault_uri, "sftp-password"),
    "db_conn_str":   notebookutils.credentials.getSecret(vault_uri, "warehouse-connection-string")
}

print("All secrets retrieved from Key Vault ✅")
# Note: never print secret values — they are auto-redacted in Fabric output

---
## 5. Variable Library Utilities — `notebookutils.variableLibrary`

Variable Libraries let you **centrally manage configuration values** (workspace IDs, lakehouse names, pipeline parameters, etc.) and reference them across notebooks — avoiding hardcoded values in code.

**Workflow:**
1. Create a Variable Library item in your Fabric workspace (e.g., `vl-visiun`)
2. Define variables and value sets in the UI
3. Access them in notebooks using `notebookutils.variableLibrary`

> ⚠️ Only supports accessing Variable Libraries **within the same workspace**. Cross-workspace access is not supported in child (reference-run) notebooks.

In [ ]:
# Load a variable library by name
vl = notebookutils.variableLibrary.getLibrary("vl-visiun")

# Access variables as object properties
print(f"Workspace Name:   {vl.Workspace_name}")
print(f"Lakehouse Name:   {vl.Lakehouse_name}")
print(f"Environment:      {vl.Environment}")

In [ ]:
# Practical use: build dynamic ABFSS paths from variable library values
vl = notebookutils.variableLibrary.getLibrary("vl-visiun")

file_path = (
    f"abfss://{vl.Workspace_name}@onelake.dfs.fabric.microsoft.com/"
    f"{vl.Lakehouse_name}.Lakehouse/Files/raw/myfile.csv"
)

df = spark.read.format("csv").option("header", "true").load(file_path)
display(df)

In [ ]:
# Access a single variable by reference string
# Format: $(/**/library_name/variable_name)
workspace_name = notebookutils.variableLibrary.get("$(/**/vl-visiun/Workspace_name)")
lakehouse_name  = notebookutils.variableLibrary.get("$(/**/vl-visiun/Lakehouse_name)")
is_prod         = notebookutils.variableLibrary.get("$(/**/vl-visiun/Is_Production)")

print(f"Workspace: {workspace_name}")
print(f"Lakehouse: {lakehouse_name}")
print(f"Is Prod:   {is_prod}")

In [ ]:
# See all available variable library methods
notebookutils.variableLibrary.help()

---
## 6. Lakehouse Utilities — `notebookutils.lakehouse`

Provides programmatic management of **Lakehouse artifacts** in a Fabric workspace — create, retrieve, update, delete, list, and trigger table loads.

In [ ]:
# Create a new Lakehouse
artifact = notebookutils.lakehouse.create(
    "lh_new_lakehouse",
    "Created via NotebookUtils"
)
print(artifact)

In [ ]:
# Create a Lakehouse with Schema support enabled
artifact = notebookutils.lakehouse.create(
    "lh_schema_enabled",
    "Lakehouse with schema support",
    {"enableSchemas": True}
)
print(artifact)

In [ ]:
# Get a Lakehouse by name
artifact = notebookutils.lakehouse.get("lh_visiun_rl")
print(artifact)

In [ ]:
# Get Lakehouse with additional properties (e.g., SQL endpoint info)
artifact = notebookutils.lakehouse.getWithProperties("lh_visiun_rl")
print(artifact)

In [ ]:
# List all Lakehouses in the current workspace
lakehouses = notebookutils.lakehouse.list()
for lh in lakehouses:
    print(lh.displayName)

In [ ]:
# List all Delta tables in a Lakehouse
tables = notebookutils.lakehouse.listTables("lh_visiun_rl")
for table in tables:
    print(table)

In [ ]:
# Load a CSV file into a Delta table (triggers a Lakehouse Load Table operation)
notebookutils.lakehouse.loadTable(
    {
        "relativePath": "Files/raw/patients.csv",  # path relative to Lakehouse root
        "pathType": "File",
        "mode": "Overwrite",                        # or "Append"
        "recursive": False,
        "formatOptions": {
            "format": "Csv",
            "header": True,
            "delimiter": ","
        }
    },
    "patients",          # target Delta table name
    "lh_visiun_rl"       # Lakehouse name
)

In [ ]:
# Rename a Lakehouse
updated = notebookutils.lakehouse.update(
    "lh_old_name",
    "lh_new_name",
    "Updated description"
)
print(updated)

In [ ]:
# Delete a Lakehouse (use with caution!)
is_deleted = notebookutils.lakehouse.delete("lh_temp_lakehouse")
print(f"Deleted: {is_deleted}")

---
## 7. Runtime Utilities — `notebookutils.runtime`

`notebookutils.runtime.context` returns a rich set of metadata about the **current session**, including workspace info, notebook identity, pipeline run context, and more.

This is extremely useful for:
- Writing environment-aware code (dev vs. prod)
- Detecting whether a notebook is running standalone or from a pipeline
- Logging and observability

In [ ]:
# Get all runtime context properties
ctx = notebookutils.runtime.context
print(ctx)

In [ ]:
# Access individual context properties
ctx = notebookutils.runtime.context

print(f"Notebook Name:        {ctx['currentNotebookName']}")
print(f"Notebook ID:          {ctx['currentNotebookId']}")
print(f"Workspace Name:       {ctx['currentWorkspaceName']}")
print(f"Workspace ID:         {ctx['currentWorkspaceId']}")
print(f"Default Lakehouse:    {ctx.get('defaultLakehouseName', 'None')}")
print(f"Default Lakehouse ID: {ctx.get('defaultLakehouseId', 'None')}")
print(f"Is Pipeline Run:      {ctx['isForPipeline']}")
print(f"Is Reference Run:     {ctx['isReferenceRun']}")

In [ ]:
# Practical pattern: environment-aware behavior based on workspace name
ctx = notebookutils.runtime.context
workspace = ctx['currentWorkspaceName']

if "dev" in workspace.lower():
    print("🔧 Running in DEVELOPMENT environment")
    log_level = "DEBUG"
elif "prod" in workspace.lower():
    print("🚀 Running in PRODUCTION environment")
    log_level = "INFO"
else:
    print(f"❓ Unknown environment: {workspace}")
    log_level = "INFO"

In [ ]:
# Detect if running from a pipeline (vs. interactively)
ctx = notebookutils.runtime.context

if ctx['isForPipeline']:
    print("Running inside a Data Factory Pipeline")
    print(f"  Activity ID: {ctx.get('activityId')}")
else:
    print("Running interactively")

if ctx['isReferenceRun']:
    print(f"  Called by parent notebook: {ctx.get('rootNotebookName')}")
    print(f"  Root Run ID: {ctx.get('rootRunId')}")

---
## 8. Session Utilities — `notebookutils.session`

Provides control over the current Spark/Python session lifecycle.

### 8.1 Stop Session — `notebookutils.session.stop()`

Stops the current interactive Spark session asynchronously, releasing all compute resources back to the pool. Useful in automation scripts where you want to free up resources after the notebook completes.

> ⚠️ Not supported in Python notebooks.

In [ ]:
# Stop the current interactive session and release compute resources
# Always call this as the LAST cell in a notebook when used
notebookutils.session.stop()

### 8.2 Restart Python Interpreter — `notebookutils.session.restartPython()`

Restarts the Python interpreter, clearing all in-memory state (imported modules, variables, etc.) without restarting the full Spark session. Useful when:
- You need a clean Python state mid-notebook
- A library has cached stale state
- Testing module changes without restarting the whole session

> ℹ️ In a reference run, this only restarts the Python interpreter of the currently referenced notebook.

In [ ]:
# Restart the Python interpreter (clears all Python state, keeps Spark session)
notebookutils.session.restartPython()

---
## 9. Quick Reference Cheat Sheet

### File System — `notebookutils.fs`

| Method | What it does |
|---|---|
| `fs.ls(path)` | List directory contents |
| `fs.mkdirs(path)` | Create directory (recursive) |
| `fs.exists(path)` | Check if file/dir exists |
| `fs.cp(src, dst, recurse)` | Copy file or directory |
| `fs.fastcp(src, dst, recurse)` | High-performance copy (AzCopy) |
| `fs.mv(src, dst, createPath, overwrite)` | Move file or directory |
| `fs.put(path, content, overwrite)` | Write string to file |
| `fs.append(path, content, create)` | Append string to file |
| `fs.head(path, maxBytes)` | Preview file content |
| `fs.rm(path, recurse)` | Delete file or directory |
| `fs.mount(source, point, config)` | Mount remote storage |
| `fs.unmount(point)` | Unmount storage |
| `fs.mounts()` | List all mount points |
| `fs.getMountPath(point)` | Get local path of mount |

### Notebook — `notebookutils.notebook`

| Method | What it does |
|---|---|
| `notebook.run(path, timeout, args, wsId)` | Run a child notebook |
| `notebook.runMultiple(DAG)` | Run notebooks in parallel/DAG |
| `notebook.validateDAG(DAG)` | Validate a DAG definition |
| `notebook.exit(value)` | Exit with a return value |
| `notebook.create(...)` | Create a notebook artifact |
| `notebook.get(name)` | Get notebook by name |
| `notebook.list()` | List all notebooks |
| `notebook.update(...)` | Rename/update a notebook |
| `notebook.delete(name)` | Delete a notebook |
| `notebook.getDefinition(name)` | Get .ipynb content |
| `notebook.updateDefinition(...)` | Update .ipynb content |

### Credentials — `notebookutils.credentials`

| Method | What it does |
|---|---|
| `credentials.getToken(audience)` | Get Entra bearer token |
| `credentials.getSecret(vault, name)` | Get secret from Key Vault |

### Variable Library — `notebookutils.variableLibrary`

| Method | What it does |
|---|---|
| `variableLibrary.getLibrary(name)` | Load full variable library |
| `variableLibrary.get(reference)` | Get single variable by ref |

### Lakehouse — `notebookutils.lakehouse`

| Method | What it does |
|---|---|
| `lakehouse.create(name, ...)` | Create a Lakehouse |
| `lakehouse.get(name)` | Get Lakehouse by name |
| `lakehouse.list()` | List all Lakehouses |
| `lakehouse.listTables(name)` | List Delta tables |
| `lakehouse.loadTable(opts, table, lh)` | Load file into Delta table |
| `lakehouse.update(old, new, ...)` | Rename Lakehouse |
| `lakehouse.delete(name)` | Delete a Lakehouse |

### Runtime & Session

| Method | What it does |
|---|---|
| `runtime.context` | Get session metadata |
| `session.stop()` | Stop Spark session |
| `session.restartPython()` | Restart Python interpreter |

In [ ]:
# Reminder: every module has a built-in help() — always your first stop!
notebookutils.fs.help()
notebookutils.notebook.help()
notebookutils.credentials.help()
notebookutils.variableLibrary.help()
notebookutils.lakehouse.help("listTables")  # help on a specific method